# Betting RL Agent — All-in-One Colab Notebook

Trains a PPO (or SAC) agent on a synthetic two-team sports betting market.
The agent learns **pure bankroll management / risk control** — market odds are
random each step so there is no predictive edge to exploit; the only learnable
signal is *when to bet* and *how much to stake*.

**Episode dynamics**
- Start: \$100 balance
- Bust: balance < \$1  →  terminal (penalty)
- Target: balance ≥ \$10,000  →  terminal (bonus)
- Timeout: 500 bets without terminal event  →  truncated

**Sections**
1. Install dependencies
2. Environment (`BettingEnv`)
3. Training (PPO / SAC via Stable-Baselines3)
4. Evaluation & plots

## 1. Install dependencies

In [ ]:
!pip install -q stable-baselines3[extra] gymnasium matplotlib

## 2. Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

import gymnasium as gym
from gymnasium import spaces

from stable_baselines3 import PPO, SAC
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor

print("All imports OK")

## 3. Betting Environment

In [ ]:
class BettingEnv(gym.Env):
    """
    Two-team win market RL environment.

    Observation space (5 features):
      [implied_prob_a, implied_prob_b, log_balance_norm, step_fraction, recent_win_rate]

    Action space (2 continuous, each in [0, 1]):
      action[0]: team selection  — <0.5 = Team A, >=0.5 = Team B
      action[1]: stake proportion — fraction of current balance to wager
                  stake < min_stake is treated as a no-bet (skip market)
    """

    metadata = {"render_modes": ["human"]}

    def __init__(
        self,
        starting_balance: float = 100.0,
        max_steps: int = 500,
        overround: float = 1.05,
        commission_rate: float = 0.05,
        min_stake: float = 1.0,
        bust_threshold: float = 1.0,
        win_threshold: float = 10_000.0,
        bust_penalty: float = -100.0,
        win_bonus: float = 100.0,
        min_true_prob: float = 0.30,
        max_true_prob: float = 0.70,
        recent_window: int = 20,
        reward_scale: float = 10.0,
        render_mode=None,
    ):
        super().__init__()

        assert 0.0 < min_true_prob < max_true_prob < 1.0
        assert commission_rate >= 0.0
        assert overround >= 1.0

        self.starting_balance = starting_balance
        self.max_steps = max_steps
        self.overround = overround
        self.commission_rate = commission_rate
        self.min_stake = min_stake
        self.bust_threshold = bust_threshold
        self.win_threshold = win_threshold
        self.bust_penalty = bust_penalty
        self.win_bonus = win_bonus
        self.min_true_prob = min_true_prob
        self.max_true_prob = max_true_prob
        self.recent_window = recent_window
        self.reward_scale = reward_scale
        self.render_mode = render_mode

        self._log_bust = np.log(bust_threshold / starting_balance)
        self._log_win  = np.log(win_threshold  / starting_balance)

        low  = np.array([0.0, 0.0, -1.0, 0.0, 0.0], dtype=np.float32)
        high = np.array([1.0, 1.0,  1.0, 1.0, 1.0], dtype=np.float32)
        self.observation_space = spaces.Box(low=low, high=high, dtype=np.float32)
        self.action_space = spaces.Box(
            low=np.zeros(2, dtype=np.float32),
            high=np.ones(2, dtype=np.float32),
            dtype=np.float32,
        )

        self.balance: float = starting_balance
        self.current_step: int = 0
        self.odds_a: float = 2.0
        self.odds_b: float = 2.0
        self.true_prob_a: float = 0.5
        self._recent_outcomes: deque = deque(maxlen=recent_window)

    # ── Market helpers ───────────────────────────────────────────────

    def _generate_market(self) -> None:
        self.true_prob_a = self.np_random.uniform(self.min_true_prob, self.max_true_prob)
        true_prob_b = 1.0 - self.true_prob_a
        self.odds_a = 1.0 / (self.true_prob_a * self.overround)
        self.odds_b = 1.0 / (true_prob_b   * self.overround)

    def _resolve_market(self) -> bool:
        return self.np_random.random() < self.true_prob_a

    # ── Gymnasium interface ──────────────────────────────────────────

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.balance = self.starting_balance
        self.current_step = 0
        self._recent_outcomes.clear()
        self._generate_market()
        return self._get_obs(), {}

    def step(self, action):
        action = np.asarray(action, dtype=np.float32)
        team_action  = float(np.clip(action[0], 0.0, 1.0))
        stake_action = float(np.clip(action[1], 0.0, 1.0))

        self._generate_market()

        old_balance = self.balance
        intended_stake = stake_action * self.balance
        no_bet = intended_stake < self.min_stake

        if no_bet:
            bet_won = False
            stake = 0.0
            selected_odds = 0.0
            bet_on_a = None
            reward = 0.0
        else:
            stake = min(intended_stake, self.balance)
            bet_on_a = team_action < 0.5
            selected_odds = self.odds_a if bet_on_a else self.odds_b

            team_a_wins = self._resolve_market()
            bet_won = (bet_on_a == team_a_wins)

            if bet_won:
                gross_profit = stake * (selected_odds - 1.0)
                net_profit = gross_profit * (1.0 - self.commission_rate)
                self.balance += net_profit
            else:
                self.balance -= stake
                self.balance = max(self.balance, 0.0)

            log_return = np.log((self.balance + 1e-8) / (old_balance + 1e-8))
            reward = float(log_return * self.reward_scale)

        self._recent_outcomes.append(1 if bet_won else 0)
        self.current_step += 1

        terminated = False
        terminal_reason = None

        if self.balance < self.bust_threshold:
            reward += self.bust_penalty
            terminated = True
            terminal_reason = "bust"
        elif self.balance >= self.win_threshold:
            reward += self.win_bonus
            terminated = True
            terminal_reason = "target_reached"

        truncated = (not terminated) and (self.current_step >= self.max_steps)
        if truncated:
            terminal_reason = "max_steps"

        info = {
            "balance": self.balance,
            "terminal_reason": terminal_reason,
            "bet_won": bet_won,
            "stake": stake,
            "selected_odds": selected_odds,
            "odds_a": self.odds_a,
            "odds_b": self.odds_b,
            "true_prob_a": self.true_prob_a,
            "no_bet": no_bet,
        }
        if terminated or truncated:
            info["final_balance"] = self.balance

        if self.render_mode == "human":
            self._render_human(action, stake, bet_won, selected_odds, terminal_reason)

        return self._get_obs(), float(reward), terminated, truncated, info

    def _get_obs(self) -> np.ndarray:
        implied_prob_a = 1.0 / self.odds_a
        implied_prob_b = 1.0 / self.odds_b
        log_bal = np.log((self.balance + 1e-8) / self.starting_balance)
        log_bal_norm = float(
            np.clip(log_bal / max(abs(self._log_bust), abs(self._log_win)), -1.0, 1.0)
        )
        step_frac = self.current_step / self.max_steps
        recent_win_rate = (
            sum(self._recent_outcomes) / len(self._recent_outcomes)
            if self._recent_outcomes else 0.5
        )
        return np.array(
            [implied_prob_a, implied_prob_b, log_bal_norm, step_frac, recent_win_rate],
            dtype=np.float32,
        )

    def _render_human(self, action, stake, bet_won, selected_odds, terminal_reason):
        team   = "A" if action[0] < 0.5 else "B"
        result = "WIN " if bet_won else "LOSE"
        print(
            f"Step {self.current_step:>4d} | "
            f"Odds A:{self.odds_a:.2f} B:{self.odds_b:.2f} | "
            f"Bet {team} @ {selected_odds:.2f} | "
            f"Stake ${stake:>8.2f} | {result} | "
            f"Balance ${self.balance:>10.2f}"
            + (f" [{terminal_reason}]" if terminal_reason else "")
        )

    def render(self):
        pass


# ── Sport presets ────────────────────────────────────────────────────
SPORT_PRESETS = {
    "afl":        {"min_true_prob": 0.30, "max_true_prob": 0.70, "overround": 1.05},
    "nrl":        {"min_true_prob": 0.30, "max_true_prob": 0.70, "overround": 1.05},
    "basketball": {"min_true_prob": 0.30, "max_true_prob": 0.70, "overround": 1.05},
    "tennis":     {"min_true_prob": 0.20, "max_true_prob": 0.80, "overround": 1.04},
    "generic":    {"min_true_prob": 0.30, "max_true_prob": 0.70, "overround": 1.05},
}

print("BettingEnv defined OK")

## 4. Quick sanity check — random agent

In [ ]:
env = BettingEnv()
obs, _ = env.reset(seed=0)
print(f"Observation shape : {obs.shape}")
print(f"Observation sample: {obs}")
print(f"Action space      : {env.action_space}")

total_reward = 0
done = False
while not done:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    done = terminated or truncated

print(f"\nRandom episode finished — reason : {info['terminal_reason']}")
print(f"Final balance   : ${info['final_balance']:.2f}")
print(f"Total reward    : {total_reward:.2f}")
env.close()

## 5. Training configuration

Adjust these settings before running the training cell.

In [ ]:
# ── Training config ──────────────────────────────────────────────────
SPORT          = "generic"   # afl | nrl | basketball | tennis | generic
ALGO           = "ppo"       # ppo | sac
TOTAL_STEPS    = 1_000_000   # increase for better convergence (try 2–5M)
N_ENVS         = 8           # parallel environments
SEED           = 42

# Episode parameters
STARTING_BAL   = 100.0
MAX_STEPS      = 500
BUST_PENALTY   = -100.0
WIN_BONUS      = 100.0
COMMISSION     = 0.05        # 5% commission on winning profit
OVERROUND      = 1.05        # 105% book

LOG_DIR        = "logs"
MODEL_DIR      = "models"
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print("Config set.")

## 6. Training

In [ ]:
# ── Monitoring callback ──────────────────────────────────────────────
class BettingMonitorCallback(BaseCallback):
    def __init__(self, log_interval=200, verbose=1):
        super().__init__(verbose)
        self.log_interval = log_interval
        self.episode_count = 0
        self.positive_episodes = 0
        self.negative_episodes = 0
        self.timeout_episodes  = 0
        self.final_balances: list = []

    def _on_step(self) -> bool:
        for done, info in zip(self.locals.get("dones", []), self.locals.get("infos", [])):
            if not done:
                continue
            self.episode_count += 1
            terminal = info.get("terminal_reason", "max_steps")
            final_balance = info.get("final_balance", 0.0)
            self.final_balances.append(final_balance)

            if terminal == "target_reached":
                self.positive_episodes += 1
            elif terminal == "bust":
                self.negative_episodes += 1
            else:
                self.timeout_episodes += 1

            self.logger.record("betting/final_balance",    final_balance)
            self.logger.record("betting/positive_rate",
                               self.positive_episodes / self.episode_count)
            self.logger.record("betting/negative_rate",
                               self.negative_episodes / self.episode_count)

            if self.verbose >= 1 and self.episode_count % self.log_interval == 0:
                n   = self.episode_count
                pos = self.positive_episodes
                neg = self.negative_episodes
                tmt = self.timeout_episodes
                avg = np.mean(self.final_balances[-self.log_interval:])
                print(
                    f"\n{'─'*55}\n"
                    f"  Episodes : {n}\n"
                    f"  Target   : {pos:>6d}  ({pos/n*100:.1f}%)\n"
                    f"  Bust     : {neg:>6d}  ({neg/n*100:.1f}%)\n"
                    f"  Timeout  : {tmt:>6d}  ({tmt/n*100:.1f}%)\n"
                    f"  Avg bal  : ${avg:.2f} (last {self.log_interval})\n"
                    f"{'─'*55}"
                )
        return True

    def summary(self):
        n = max(self.episode_count, 1)
        return dict(
            total=self.episode_count,
            positive=self.positive_episodes,
            negative=self.negative_episodes,
            timeout=self.timeout_episodes,
            positive_rate=self.positive_episodes / n,
            negative_rate=self.negative_episodes / n,
            mean_final_balance=float(np.mean(self.final_balances)) if self.final_balances else 0.0,
        )


# ── Build envs ───────────────────────────────────────────────────────
preset = SPORT_PRESETS.get(SPORT, SPORT_PRESETS["generic"]).copy()
preset.update(dict(
    commission_rate=COMMISSION,
    overround=OVERROUND,
    starting_balance=STARTING_BAL,
    max_steps=MAX_STEPS,
    bust_penalty=BUST_PENALTY,
    win_bonus=WIN_BONUS,
))

def _make_env():
    return Monitor(BettingEnv(**preset))

train_env = make_vec_env(_make_env, n_envs=N_ENVS, seed=SEED)
eval_env  = make_vec_env(_make_env, n_envs=1,      seed=SEED + 9999)

# ── Callbacks ────────────────────────────────────────────────────────
monitor_cb = BettingMonitorCallback(log_interval=200, verbose=1)
eval_cb = EvalCallback(
    eval_env,
    best_model_save_path=MODEL_DIR,
    log_path=LOG_DIR,
    eval_freq=max(50_000 // N_ENVS, 1),
    n_eval_episodes=100,
    deterministic=True,
    verbose=0,
)

# ── Model ────────────────────────────────────────────────────────────
if ALGO == "ppo":
    model = PPO(
        policy="MlpPolicy",
        env=train_env,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=64,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.01,
        vf_coef=0.5,
        max_grad_norm=0.5,
        policy_kwargs=dict(net_arch=[128, 128]),
        tensorboard_log=LOG_DIR,
        seed=SEED,
        verbose=0,
    )
else:  # SAC
    model = SAC(
        policy="MlpPolicy",
        env=train_env,
        learning_rate=3e-4,
        buffer_size=1_000_000,
        batch_size=256,
        gamma=0.99,
        tau=0.005,
        ent_coef="auto",
        policy_kwargs=dict(net_arch=[128, 128]),
        tensorboard_log=LOG_DIR,
        seed=SEED,
        verbose=0,
    )

print(f"Training {ALGO.upper()} for {TOTAL_STEPS:,} steps across {N_ENVS} envs...")
model.learn(total_timesteps=TOTAL_STEPS, callback=[monitor_cb, eval_cb], progress_bar=True)

model.save(os.path.join(MODEL_DIR, "final_model"))
train_env.close()
eval_env.close()

s = monitor_cb.summary()
print(f"\n{'='*55}")
print("  Training Complete")
print(f"{'='*55}")
print(f"  Total episodes : {s['total']:,}")
print(f"  Target reached : {s['positive']:,}  ({s['positive_rate']*100:.2f}%)")
print(f"  Bust           : {s['negative']:,}  ({s['negative_rate']*100:.2f}%)")
print(f"  Timeout        : {s['timeout']:,}")
print(f"  Mean final bal : ${s['mean_final_balance']:.2f}")
print(f"{'='*55}")

## 7. Evaluation

In [ ]:
# ── Config ───────────────────────────────────────────────────────────
MODEL_PATH  = os.path.join(MODEL_DIR, "best_model.zip")  # or "final_model.zip"
N_EVAL_EPS  = 500
EVAL_SEED   = 0

# ── Load ─────────────────────────────────────────────────────────────
AlgoCls = PPO if ALGO == "ppo" else SAC
eval_model = AlgoCls.load(MODEL_PATH)
print(f"Loaded: {MODEL_PATH}")

# ── Run episodes ─────────────────────────────────────────────────────
eval_env_single = BettingEnv(**preset)
rng = np.random.default_rng(EVAL_SEED)

outcomes        = {"target_reached": 0, "bust": 0, "max_steps": 0}
final_balances  = []
episode_lengths = []
team_actions    = []
stake_actions   = []
sample_trajs    = []
N_TRAJ          = min(50, N_EVAL_EPS)

for ep in range(N_EVAL_EPS):
    obs, _ = eval_env_single.reset(seed=int(rng.integers(1 << 30)))
    done = False
    bal_hist = [STARTING_BAL]
    steps = 0

    while not done:
        action, _ = eval_model.predict(obs, deterministic=True)
        obs, _, terminated, truncated, info = eval_env_single.step(action)
        done = terminated or truncated
        team_actions.append(float(action[0]))
        stake_actions.append(float(action[1]))
        bal_hist.append(info["balance"])
        steps += 1

    terminal_reason = info.get("terminal_reason", "max_steps")
    final_bal = info.get("final_balance", info["balance"])
    outcomes[terminal_reason] = outcomes.get(terminal_reason, 0) + 1
    final_balances.append(final_bal)
    episode_lengths.append(steps)
    if ep < N_TRAJ:
        sample_trajs.append(bal_hist)

    if (ep + 1) % 100 == 0:
        print(f"  {ep+1}/{N_EVAL_EPS} episodes done")

eval_env_single.close()

n   = N_EVAL_EPS
pos = outcomes.get("target_reached", 0)
neg = outcomes.get("bust", 0)
tmt = outcomes.get("max_steps", 0)

print(f"\n{'='*55}")
print(f"  Evaluation Results  ({n} episodes)")
print(f"{'='*55}")
print(f"  Target reached : {pos:>5d}  ({pos/n*100:.2f}%)")
print(f"  Bust           : {neg:>5d}  ({neg/n*100:.2f}%)")
print(f"  Timeout        : {tmt:>5d}  ({tmt/n*100:.2f}%)")
print(f"  Mean balance   : ${np.mean(final_balances):.2f}")
print(f"  Median balance : ${np.median(final_balances):.2f}")
print(f"  Std balance    : ${np.std(final_balances):.2f}")
print(f"  Mean ep length : {np.mean(episode_lengths):.1f} bets")
print(f"{'='*55}")

## 8. Plots

In [ ]:
assert 'sample_trajs' in dir() and len(sample_trajs) > 0, \
    "Run Cell 7 (Evaluation) before plotting."

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ── Balance trajectories ─────────────────────────────────────────────
ax = axes[0, 0]
for traj in sample_trajs:
    ax.plot(traj, alpha=0.25, linewidth=0.7, color="steelblue")
ax.axhline(preset.get("win_threshold", 10_000), color="green",  lw=1.5, ls="--", label=f"Win target")
ax.axhline(preset.get("bust_threshold", 1.0),   color="red",    lw=1.5, ls="--", label=f"Bust")
ax.axhline(STARTING_BAL,                         color="grey",   lw=1.0, ls=":",  label=f"Start")
ax.set_yscale("log")
ax.set_xlabel("Bet number")
ax.set_ylabel("Balance ($, log scale)")
ax.set_title(f"Balance Trajectories — {len(sample_trajs)} episodes")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# ── Final balance distribution (log scale) ───────────────────────────
ax = axes[0, 1]
log_bals = np.log10(np.maximum(final_balances, 0.01))
ax.hist(log_bals, bins=50, color="darkorange", edgecolor="white", alpha=0.8)
for exp, label in [(0, "$1"), (2, "$100"), (3, "$1k"), (4, "$10k")]:
    ax.axvline(exp, color="grey", lw=0.8, ls="--")
    ax.text(exp + 0.05, ax.get_ylim()[1] * 0.85, label, fontsize=8, color="grey")
ax.set_xlabel("log₁₀(Final Balance)")
ax.set_ylabel("Episode count")
ax.set_title("Final Balance Distribution (log scale)")
ax.grid(True, alpha=0.3)

# ── Episode outcome pie ──────────────────────────────────────────────
ax = axes[1, 0]
pie_labels, pie_sizes, pie_colours = [], [], []
for key, col in [("target_reached", "seagreen"), ("bust", "tomato"), ("max_steps", "steelblue")]:
    v = outcomes.get(key, 0)
    if v > 0:
        pie_labels.append(f"{key.replace('_',' ').title()} ({v})")
        pie_sizes.append(v)
        pie_colours.append(col)
ax.pie(pie_sizes, labels=pie_labels, colors=pie_colours, autopct="%1.1f%%",
       startangle=90, textprops={"fontsize": 10})
ax.set_title("Episode Outcome Distribution")

# ── Stake distribution ───────────────────────────────────────────────
ax = axes[1, 1]
ax.hist(stake_actions, bins=50, color="mediumpurple", edgecolor="white", alpha=0.8)
ax.set_xlabel("Stake proportion (fraction of balance)")
ax.set_ylabel("Frequency")
ax.set_title(
    f"Stake Proportion Distribution\n"
    f"Mean: {np.mean(stake_actions):.3f}   Median: {np.median(stake_actions):.3f}"
)
ax.grid(True, alpha=0.3)

fig.suptitle(
    f"Betting RL Agent — {ALGO.upper()} / {SPORT} / {TOTAL_STEPS:,} steps",
    fontsize=14, y=1.01
)
fig.tight_layout()
plt.show()

## 9. Random agent benchmark

Compare the trained agent against a random baseline to confirm it is learning something meaningful.

In [ ]:
bench_env = BettingEnv(**preset)
rng2 = np.random.default_rng(EVAL_SEED)
rand_outcomes = {"target_reached": 0, "bust": 0, "max_steps": 0}
rand_balances = []

for _ in range(N_EVAL_EPS):
    obs, _ = bench_env.reset(seed=int(rng2.integers(1 << 30)))
    done = False
    while not done:
        action = bench_env.action_space.sample()
        obs, _, terminated, truncated, info = bench_env.step(action)
        done = terminated or truncated
    t = info.get("terminal_reason", "max_steps")
    rand_outcomes[t] = rand_outcomes.get(t, 0) + 1
    rand_balances.append(info.get("final_balance", info["balance"]))

bench_env.close()

r_pos = rand_outcomes.get("target_reached", 0)
r_neg = rand_outcomes.get("bust", 0)

print(f"{'─'*55}")
print(f"  {'Metric':<28} {'Trained':>10} {'Random':>10}")
print(f"{'─'*55}")
print(f"  {'Target reached rate':<28} {pos/n*100:>9.2f}% {r_pos/n*100:>9.2f}%")
print(f"  {'Bust rate':<28} {neg/n*100:>9.2f}% {r_neg/n*100:>9.2f}%")
print(f"  {'Mean final balance':<28} ${np.mean(final_balances):>9.2f} ${np.mean(rand_balances):>9.2f}")
print(f"  {'Median final balance':<28} ${np.median(final_balances):>9.2f} ${np.median(rand_balances):>9.2f}")
print(f"{'─'*55}")